# Description

In this notebook, we benchmark ComplexEQL algorithm on the Korns benchmarks.

In [1]:
from __future__ import annotations

# ============================================================
# CPU limiting (target: <= 8 CPU cores total)
#   - Use 8 worker processes
#   - Force 1 thread per process for BLAS/OpenMP + PyTorch
# ============================================================
import os

TARGET_CPUS = 8
THREADS_PER_WORKER = 1  # 8 procs × 1 thread ≈ 8 CPUs

os.environ["OMP_NUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["MKL_NUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["OPENBLAS_NUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(THREADS_PER_WORKER)
os.environ["NUMEXPR_NUM_THREADS"] = str(THREADS_PER_WORKER)

from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import sympy as sp
import h5py
import torch
from torch.utils.data import TensorDataset, DataLoader

from config.korns_config import BENCH, FEATURE_NAMES, CEQL_TRAIN, CEQL
from src.korns_core import (
    RunConfig,
    BenchmarkRow,
    train_test_split,
    init_results_csv,
    append_results_csv_row,
    zlib_crc32,
)
from src.metrics import Metrics, compute_metrics

from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train
from src.sympy_utils import filter_imaginary_part


def complex_l1_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    diff = pred - target
    return diff.real.abs().mean() + diff.imag.abs().mean()


def _get_algo_seed(algo_name: str) -> int:
    return int(np.uint32(zlib_crc32(algo_name.encode("utf-8"))))


def _load_one_korns_problem(hdf5_path: str, pid: str) -> Tuple[np.ndarray, np.ndarray, sp.Expr]:
    with h5py.File(hdf5_path, "r") as f:
        grp = f[pid]
        X = np.asarray(grp["X"][:], dtype=np.float64)
        y = np.asarray(grp["y"][:], dtype=np.float64).reshape(-1)
        if "expr_srepr" in grp.attrs:
            expr_gt = sp.sympify(grp.attrs["expr_srepr"])
        else:
            expr_gt = sp.sympify(grp.attrs["expr_str"])
    return X, y, expr_gt


def _ceql_fit_predict_one_seed(
    *,
    hdf5_path: str,
    pid: str,
    cfg: RunConfig,
    run_id: int,
    feature_names: List[str],
    algo_name: str = "complexeql",
) -> BenchmarkRow:
    # Safe to call in worker; do NOT call set_num_interop_threads here.
    torch.set_num_threads(THREADS_PER_WORKER)

    # Load only what this worker needs
    X, y, expr_gt = _load_one_korns_problem(hdf5_path, pid)

    # Split (per-problem, deterministic)
    split_seed = cfg.split_seed + cfg.per_problem_seed_offset + int(pid[1:])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_size, seed=split_seed
    )

    # Run seed (per pid/algo/run_id), wrapped to uint32 domain
    algo_seed = _get_algo_seed(algo_name)
    run_seed = (
        cfg.split_seed
        + cfg.per_problem_seed_offset * int(pid[1:])
        + cfg.algo_seed_offset * algo_seed
        + cfg.run_seed_offset * run_id
    )
    run_seed = int(run_seed % (2**32))
    set_seed(run_seed)

    # -------- CEQL training --------
    device = torch.device(getattr(CEQL_TRAIN, "device", "cpu"))

    Xtr = torch.from_numpy(np.asarray(X_train, dtype=np.float32))
    Xte = torch.from_numpy(np.asarray(X_test, dtype=np.float32))

    ytr_real = torch.from_numpy(np.asarray(y_train, dtype=np.float32).reshape(-1, 1))
    ytr = torch.complex(ytr_real, torch.zeros_like(ytr_real))

    batch_size = int(getattr(CEQL_TRAIN, "train_batch_size", 4096))
    dataloader = DataLoader(
        TensorDataset(Xtr, ytr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    # Best-effort align input dimensionality for symbolics/readout
    n_in = int(Xtr.shape[1])
    try:
        CEQL.n_input_fields = n_in
    except Exception:
        pass

    model = ComplexEQL(CEQL).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(getattr(CEQL_TRAIN, "lr", 1e-3)))

    scheduler = None
    if getattr(CEQL_TRAIN, "scheduler", None) == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, **getattr(CEQL_TRAIN, "schedulerparams", {})
        )

    model, logs = train(
        model=model,
        dataloader=dataloader,
        optimizer=optimizer,
        loss_fn=complex_l1_loss,
        cfg=CEQL_TRAIN,
        device=device,
        scheduler=scheduler,
    )

    # Predict (real part for metrics)
    model.eval()
    with torch.no_grad():
        pred_c = model(Xte.to(device))
        y_pred = pred_c.real.detach().cpu().numpy().reshape(-1).astype(np.float64)

    # Symbolic expression
    # try:
    #     symbols = sp.symbols(" ".join(feature_names[:n_in]))
    #     expr_pred = model.get_symbolic_expression(list(symbols), rounding_decimals=2)
    #     expr_pred = filter_imaginary_part(expr_pred)
    # except Exception:
    #     expr_pred = None
    expr_pred = None

    # Metrics
    m: Metrics = compute_metrics(
        y_true=y_test,
        y_pred=y_pred,
        expr_gt=expr_gt,
        expr_pred=expr_pred,
        feature_names=feature_names,
    )

    expr_pred_str = str(expr_pred) if expr_pred is not None else ""

    extra: Dict[str, Any] = {
        "run_seed": int(run_seed),
        "device": str(device),
        "batch_size": int(batch_size),
        "threads_per_worker": int(THREADS_PER_WORKER),
    }
    try:
        if isinstance(logs, tuple) and len(logs) >= 5:
            imag_losses = logs[3]
            data_losses = logs[4]
            if hasattr(imag_losses, "__len__") and len(imag_losses) > 0:
                extra["final_imag_loss"] = float(imag_losses[-1])
            if hasattr(data_losses, "__len__") and len(data_losses) > 0:
                extra["final_data_loss"] = float(data_losses[-1])
    except Exception:
        pass

    return BenchmarkRow(
        pid=pid,
        algo=algo_name,
        run_id=run_id,
        nlse=m.nlse,
        term_precision=m.term_precision,
        term_recall=m.term_recall,
        term_f1=m.term_f1,
        expr_str=expr_pred_str,
        expr_gt_str=str(expr_gt),
        extra=extra,
    )


def run_ceql_parallel_in_this_script_only(
    *,
    hdf5_path: str,
    cfg: RunConfig,
    pids: List[str],
    n_runs: int,
    feature_names: List[str],
    results_csv_path: str,
    max_workers: Optional[int] = None,
) -> List[BenchmarkRow]:
    init_results_csv(results_csv_path, BenchmarkRow)

    rows: List[BenchmarkRow] = []
    tasks = [(pid, run_id) for pid in pids for run_id in range(n_runs)]

    if max_workers is None:
        max_workers = TARGET_CPUS

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futs = [
            ex.submit(
                _ceql_fit_predict_one_seed,
                hdf5_path=hdf5_path,
                pid=pid,
                cfg=cfg,
                run_id=run_id,
                feature_names=feature_names,
                algo_name="complexeql",
            )
            for (pid, run_id) in tasks
        ]

        for fut in as_completed(futs):
            row = fut.result()
            rows.append(row)
            append_results_csv_row(results_csv_path, row)
            print(f"[DONE] {row.pid} run_id={row.run_id} nlse={row.nlse:.3e}")

    rows.sort(key=lambda r: (int(r.pid[1:]), r.run_id))
    return rows


# ============================================================
# Main
# ============================================================

cfg = RunConfig(
    hdf5_path=BENCH.hdf5_path,
    test_size=BENCH.test_size,
    split_seed=BENCH.split_seed,
    per_problem_seed_offset=BENCH.per_problem_seed_offset,
    algo_seed_offset=BENCH.algo_seed_offset,
    run_seed_offset=BENCH.run_seed_offset,
)

with h5py.File(cfg.hdf5_path, "r") as f:
    pids = sorted(list(f.keys()), key=lambda k: int(k[1:]))

CEQL_RESULTS_CSV = "korns_complexeql_benchmark_results.csv"

rows = run_ceql_parallel_in_this_script_only(
    hdf5_path=cfg.hdf5_path,
    cfg=cfg,
    pids=pids,
    n_runs=BENCH.n_runs,
    feature_names=FEATURE_NAMES,
    results_csv_path=CEQL_RESULTS_CSV,
    max_workers=TARGET_CPUS,  # 8 processes, each 1 thread
)

print(f"Saved incremental results to: {CEQL_RESULTS_CSV}")


Random seed set as 263591608Random seed set as 260592608

Random seed set as 264591608Random seed set as 262591608Random seed set as 260591608


Random seed set as 261591608
[Phase 1 | Epoch 1] total=4.5990e+14, data=4.5990e+14, sparsity_reg=0.0000e+00, imag_w=6.2846e-03, alpha=0.000e+00, imag_coeff=1.000e-04, lr=1.00e-03
[Phase 1 | Epoch 1] total=5.9612e+14, data=5.9612e+14, sparsity_reg=0.0000e+00, imag_w=6.1814e-03, alpha=0.000e+00, imag_coeff=1.000e-04, lr=1.00e-03
[Phase 1 | Epoch 1] total=6.6004e+14, data=6.6004e+14, sparsity_reg=0.0000e+00, imag_w=6.2959e-03, alpha=0.000e+00, imag_coeff=1.000e-04, lr=1.00e-03
[Phase 1 | Epoch 1] total=1.2560e+15, data=1.2560e+15, sparsity_reg=0.0000e+00, imag_w=6.2015e-03, alpha=0.000e+00, imag_coeff=1.000e-04, lr=1.00e-03
[Phase 1 | Epoch 1] total=7.1908e+14, data=7.1908e+14, sparsity_reg=0.0000e+00, imag_w=6.2952e-03, alpha=0.000e+00, imag_coeff=1.000e-04, lr=1.00e-03[Phase 1 | Epoch 1] total=8.9551e+14, data=8.9551e+14, sparsity_reg=0.0000e+0